In [1]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

from sklearn.preprocessing import OneHotEncoder

import yaml

# Load HCM data

In [2]:
all_time_df = pd.read_csv("../data/preprocess/all_train.csv")
all_time_df.head()

,_id,segment_id,date,weekday,period,LOS,s_node_id,e_node_id,length,street_id,...,long_snode,lat_snode,long_enode,lat_enode,street_name,street_type,status_id,updated_at,velocity,time_bucket
0,0,26,2021-04-16,4,period_0_30,A,366428456,366416066,0.116,32575820,...,106.768732,10.841506,106.769254,10.842422,Nguyễn Văn Bá,tertiary,89252,2021-04-16 00:55:31.333000+00:00,117,2021-04-16 00:30:00+00:00
1,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,892,2020-08-02 23:56:28.019000+00:00,34,2020-08-02 23:30:00+00:00
2,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,4501,2020-08-02 23:59:28.137000+00:00,26,2020-08-02 23:30:00+00:00
3,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,8110,2020-08-03 00:02:28.070000+00:00,2,2020-08-03 00:00:00+00:00
4,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,0.026,32575862,...,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary,11719,2020-08-03 00:05:28.204000+00:00,35,2020-08-03 00:00:00+00:00


In [3]:
train_df = pd.read_csv("../data/raw/train.csv")
train_df.head()

,_id,segment_id,date,weekday,period,LOS,s_node_id,e_node_id,length,street_id,max_velocity,street_level,street_name,street_type,long_snode,lat_snode,long_enode,lat_enode
0,0,26,2021-04-16,4,period_0_30,A,366428456,366416066,116,32575820,NaN,4,Nguyễn Văn Bá,tertiary,106.768732,10.841506,106.769254,10.842422
1,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,26,32575862,NaN,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808
2,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,26,32575862,NaN,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808
3,3,67,2021-03-09,1,period_9_30,B,366403668,5755066033,7,32575862,NaN,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771
4,4,67,2021-03-23,1,period_9_30,B,366403668,5755066033,7,32575862,NaN,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771


In [4]:
all_df = pd.read_csv("../data/preprocess/all.csv")
all_df.head()

,_id,segment_id,date,weekday,period,LOS,s_node_id,e_node_id,length,street_id,max_velocity,street_level,segment_name,segment_type,long_snode,lat_snode,long_enode,lat_enode,street_name,street_type
0,0,26,2021-04-16,4,period_0_30,A,366428456,366416066,0.116,32575820,40.0,4,Nguyễn Văn Bá,tertiary,106.768732,10.841506,106.769254,10.842422,Nguyễn Văn Bá,tertiary
1,1,33,2020-08-02,6,period_23_30,C,366469460,3792257828,0.026,32575862,40.0,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary
2,2,33,2020-08-03,0,period_0_00,D,366469460,3792257828,0.026,32575862,40.0,3,Đường số 5,secondary,106.761957,10.878650,106.762143,10.878808,Đường số 5,secondary
3,3,67,2021-03-09,1,period_9_30,B,366403668,5755066033,0.007,32575862,40.0,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771,Đường số 5,secondary
4,4,67,2021-03-23,1,period_9_30,B,366403668,5755066033,0.007,32575862,40.0,3,Đường số 5,secondary,106.768412,10.880817,106.768461,10.880771,Đường số 5,secondary


In [5]:
nodes = sorted(
    set(all_df["s_node_id"]) |
    set(all_df["e_node_id"])
)

In [6]:
nodes_df = pd.read_csv("../data/raw/nodes.csv")
nodes_df = nodes_df[nodes_df["_id"].isin(nodes)]
print(nodes_df.shape)

(11314, 3)


In [7]:
streets = sorted(set(all_df["street_id"]))
print(len(streets))

1967


In [8]:
streets_df = pd.read_csv("../data/raw/streets.csv")
streets_df = streets_df[streets_df["_id"].isin(streets)]
print(streets_df.shape)

(1967, 5)


In [9]:
segments_df = all_df[["_id", "street_id", "s_node_id", "e_node_id", "length", "segment_type"]]
segments_df.head()

,_id,street_id,s_node_id,e_node_id,length,segment_type
0,0,32575820,366428456,366416066,0.116,tertiary
1,1,32575862,366469460,3792257828,0.026,secondary
2,2,32575862,366469460,3792257828,0.026,secondary
3,3,32575862,366403668,5755066033,0.007,secondary
4,4,32575862,366403668,5755066033,0.007,secondary


# Load OSM

In [10]:
with open("../data/raw/osm_train_2019_01_03.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [11]:
osm_elements_df = pd.json_normalize(osm_data["elements"])
osm_elements_df.head()

,type,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## OSM Node

In [12]:
osm_nodes_df = osm_elements_df[osm_elements_df["type"] == "node"]
osm_nodes_df = osm_nodes_df[osm_nodes_df["id"].isin(nodes)]
osm_nodes_df = osm_nodes_df.drop(columns=["type"])
print(osm_nodes_df.shape)
osm_nodes_df.head()

(11314, 129)


,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,tags.note,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
1,366367392,10.775732,106.614032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,366367451,10.793792,106.695366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,366367839,10.807689,106.664528,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,366367847,10.814589,106.671636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,366368010,10.763042,106.644773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## OSM Way

In [13]:
osm_ways_df = osm_elements_df[osm_elements_df["type"] == "way"]
osm_ways_df = osm_ways_df[osm_ways_df["id"].isin(streets)]
osm_ways_df = osm_ways_df.drop(columns=["type"])
print(osm_ways_df.shape)
osm_ways_df.head()

(1967, 129)


,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,tags.note,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
3066,32577244,NaN,NaN,tertiary,NaN,NaN,NaN,NaN,Tôn Thất Hiệp,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3067,32577484,NaN,NaN,tertiary,NaN,NaN,NaN,NaN,Gò Dưa,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3068,32577508,NaN,NaN,tertiary,NaN,NaN,NaN,NaN,Kha Vạn Cân,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3071,32578529,NaN,NaN,tertiary,NaN,NaN,NaN,NaN,Tân Chánh Hiệp 39,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3072,32578530,NaN,NaN,unclassified,NaN,NaN,NaN,NaN,THOI TAM THÔN 5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
data = []

for _, row in osm_ways_df.iterrows():
    way_id = row['id']
    nodes = row['nodes']
    if isinstance(nodes, list) and len(nodes) >= 2:
        for i in range(len(nodes)-1):
            data.append({
                'street_id': way_id,
                's_node_id': nodes[i],      # from
                'e_node_id': nodes[i+1]     # to
            })

osm_edges_df = pd.DataFrame(data)
print(osm_edges_df.shape)
osm_edges_df.head()

(25092, 3)


,street_id,s_node_id,e_node_id
0,32577244,366445888,2434647895
1,32577244,2434647895,4929127560
2,32577244,4929127560,366388735
3,32577244,366388735,366474637
4,32577244,366474637,5778485990


# Kết hợp data

## Node

In [15]:
combine_nodes_df = nodes_df.merge(
    osm_nodes_df,
    left_on="_id",
    right_on="id",
    how="inner"
)
combine_nodes_df = combine_nodes_df.drop(columns=["_id"])
print(combine_nodes_df.shape)
combine_nodes_df.head()

(11314, 131)


,long,lat_x,id,lat_y,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
0,106.614032,10.775732,366367392,10.775732,106.614032,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,106.695366,10.793792,366367451,10.793792,106.695366,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,106.664528,10.807689,366367839,10.807689,106.664528,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,106.671636,10.814589,366367847,10.814589,106.671636,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,106.644773,10.763042,366368010,10.763042,106.644773,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Way

In [16]:
combine_ways_df = streets_df.merge(
    osm_ways_df,
    left_on="_id",
    right_on="id",
    how="inner"
)
combine_ways_df = combine_ways_df.drop(columns=["_id"])
print(combine_ways_df.shape)
combine_ways_df.head()

(1967, 133)


,level,max_velocity,name,type,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
0,4,NaN,Nguyễn Văn Bá,tertiary,32575820,NaN,NaN,tertiary,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,NaN,Đường số 5,secondary,32575862,NaN,NaN,secondary,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,40.0,Châu Văn Liêm,secondary,32575864,NaN,NaN,secondary,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,Lê Văn Thịnh,unclassified,32575869,NaN,NaN,unclassified,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,NaN,Tân Phú,primary_link,32575935,NaN,NaN,primary_link,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# One-hot

## Node

In [17]:
node_connectivity_tags = [
    "tags.railway",
    "tags.junction",
    "tags.crossing",
    "tags.highway"
]

In [18]:
combine_nodes_df["tags.railway"] = combine_nodes_df["tags.railway"].fillna("no")

In [19]:
combine_nodes_df["tags.junction"] = combine_nodes_df["tags.junction"].fillna("no")

In [20]:
combine_nodes_df["tags.crossing"] = combine_nodes_df["tags.crossing"].fillna("no")

In [21]:
combine_nodes_df["tags.highway"] = combine_nodes_df["tags.highway"].fillna("no")

In [22]:
for tag in node_connectivity_tags:
    node_perc = (combine_nodes_df[tag].value_counts(normalize=True) * 100).round(2)
    print(node_perc)
    print("_"*50)

tags.railway
no                99.86
level_crossing     0.12
crossing           0.01
station            0.01
Name: proportion, dtype: float64
__________________________________________________
tags.junction
no     99.99
yes     0.01
Name: proportion, dtype: float64
__________________________________________________
tags.crossing
no                 99.93
zebra               0.05
traffic_signals     0.02
Name: proportion, dtype: float64
__________________________________________________
tags.highway
no                 94.10
bus_stop            3.87
traffic_signals     1.89
crossing            0.14
Name: proportion, dtype: float64
__________________________________________________


In [23]:
node_public_tags = [
    "tags.bus",
    "tags.ferry",
]

In [24]:
combine_nodes_df["tags.bus"] = combine_nodes_df["tags.bus"].fillna("no")

In [25]:
combine_nodes_df["tags.ferry"] = combine_nodes_df["tags.ferry"].fillna("no")

In [26]:
for tag in node_public_tags:
    node_perc = (combine_nodes_df[tag].value_counts(normalize=True) * 100).round(2)
    print(node_perc)
    print("_"*50)

tags.bus
no     99.99
yes     0.01
Name: proportion, dtype: float64
__________________________________________________
tags.ferry
no    100.0
Name: proportion, dtype: float64
__________________________________________________


In [27]:
# Bỏ qua ferry vì toàn null
node_public_tags = [x for x in node_public_tags if not (x == "tags.ferry")]
node_attr_tags = node_connectivity_tags + node_public_tags

In [28]:
node_oh_encoder = OneHotEncoder(
    drop='first',        
    sparse_output=False,
)

node_encoded_array = node_oh_encoder.fit_transform(combine_nodes_df[node_attr_tags])
node_oh_encoder.categories_

[array(['crossing', 'level_crossing', 'no', 'station'], dtype=object),
 array(['no', 'yes'], dtype=object),
 array(['no', 'traffic_signals', 'zebra'], dtype=object),
 array(['bus_stop', 'crossing', 'no', 'traffic_signals'], dtype=object),
 array(['no', 'yes'], dtype=object)]

In [29]:
node_oh_encoded_df = pd.DataFrame(
    node_encoded_array, 
    columns=node_oh_encoder.get_feature_names_out(),
    index=combine_nodes_df.index
)

node_oh_encoded_df.insert(0, "id", combine_nodes_df["id"])
node_oh_encoded_df.head()

,id,tags.railway_level_crossing,tags.railway_no,tags.railway_station,tags.junction_yes,tags.crossing_traffic_signals,tags.crossing_zebra,tags.highway_crossing,tags.highway_no,tags.highway_traffic_signals,tags.bus_yes
0,366367392,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,366367451,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,366367839,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,366367847,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,366368010,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Way

### Structure

In [30]:
way_structure_tags = [
    "tags.surface",
    "tags.layer",
    "tags.bridge",
    "tags.lanes",
    "tags.oneway"
]

In [31]:
# Fill nan bằng "paved" nếu đường có type rõ ràng
street_types = streets_df["type"].unique().tolist()
paved_surface = [t for t in street_types if t != "unclassified"]

paved_mask = (
    combine_ways_df["type"].isin(paved_surface) & 
    combine_ways_df["tags.surface"].isna()
)
combine_ways_df.loc[paved_mask, "tags.surface"] = "paved"
combine_ways_df["tags.surface"] = combine_ways_df["tags.surface"].fillna("unpaved")

In [32]:
# Fill NaN bằng 0 cho layer
combine_ways_df["tags.layer"] = combine_ways_df["tags.layer"].fillna(0)

In [33]:
combine_ways_df["tags.bridge"] = combine_ways_df["tags.bridge"].fillna("no")

In [34]:
combine_ways_df["tags.tunnel"] = combine_ways_df["tags.tunnel"].fillna("no")

In [35]:
# Fill nan bằng "2" nếu đường có type rõ ràng, còn không là 1 cho lanes
paved_mask = (
    combine_ways_df["type"].isin(paved_surface) & 
    combine_ways_df["tags.lanes"].isna()
)
combine_ways_df.loc[paved_mask, "tags.lanes"] = 2
combine_ways_df["tags.lanes"] = combine_ways_df["tags.lanes"].fillna(1).infer_objects(copy=False)

In [36]:
# Oneway chỉ mang 2 giá trị yes hoặc no nhưng có 2 điểm mang "no;yes"
combine_ways_df["tags.oneway"] = combine_ways_df["tags.oneway"].apply(
    lambda x: "no" if not (x in ["yes", "no"]) else x
)
combine_ways_df["tags.oneway"] = combine_ways_df["tags.oneway"].fillna("no")

In [37]:
combine_ways_df["tags.oneway"] .unique()

array(['no', 'yes'], dtype=object)

In [38]:
for tag in way_structure_tags:
    node_perc = (combine_ways_df[tag].value_counts(normalize=True) * 100).round(2)
    print(node_perc)
    print("_"*50)

tags.surface
paved       86.27
unpaved      8.13
asphalt      5.39
concrete     0.10
dirt         0.10
Name: proportion, dtype: float64
__________________________________________________
tags.layer
0     90.44
1      7.12
2      1.42
5      0.56
3      0.36
-1     0.10
Name: proportion, dtype: float64
__________________________________________________
tags.bridge
no     91.31
yes     8.69
Name: proportion, dtype: float64
__________________________________________________
tags.lanes
2    86.07
1     8.24
2     2.34
3     1.98
1     0.61
4     0.41
6     0.25
5     0.05
8     0.05
Name: proportion, dtype: float64
__________________________________________________
tags.oneway
yes    65.48
no     34.52
Name: proportion, dtype: float64
__________________________________________________


### Controls & Restrictions

In [39]:
way_control_tags = [
    #"tags.maxspeed",
    "tags.minspeed",
    "tags.motorroad"
]

Fill tốc độ theo quy định của VN [Quy định tốc độ giao thông đường bộ](https://baochinhphu.vn/quy-dinh-ve-toc-do-toi-da-cua-xe-co-gioi-ap-dung-tu-01-01-2025-102241127101044466.htm)

#### Max speed

In [40]:
combine_ways_df["tags.maxspeed"].unique()

array([nan, '40', '50', '20', '60', '80', '45', '10', '70', '40,50,60',
       '120', '30', '100'], dtype=object)

In [41]:
combine_ways_df["tags.maxspeed"] = combine_ways_df["tags.maxspeed"].fillna(-1)

In [42]:
combine_ways_df["max_velocity"].unique()

array([ nan,  40.,  50.,  20.,  60.,  80.,  45.,  10.,  70., 120.,  30.,
       100.])

In [43]:
combine_ways_df["max_velocity"] = combine_ways_df["max_velocity"].fillna(-1)

In [44]:
conflict_maxspeed_df = combine_ways_df[["id", "tags.maxspeed", "max_velocity"]]
conflict_maxspeed_df.head()

,id,tags.maxspeed,max_velocity
0,32575820,-1,-1.0
1,32575862,-1,-1.0
2,32575864,40,40.0
3,32575869,-1,-1.0
4,32575935,-1,-1.0


In [45]:
conflict_maxspeed_df[conflict_maxspeed_df["tags.maxspeed"] == "40,50,60"]

,id,tags.maxspeed,max_velocity
801,227746179,"40,50,60",-1.0
1318,431068520,"40,50,60",-1.0


In [46]:
conflict_mask = conflict_maxspeed_df["tags.maxspeed"] == "40,50,60"
conflict_maxspeed_df.iloc[conflict_mask, 1] = 40

In [47]:
conflict_maxspeed_df["tags.maxspeed"] = conflict_maxspeed_df["tags.maxspeed"].astype(float)
conflict_maxspeed_df["max_velocity"] = conflict_maxspeed_df["max_velocity"].astype(float)

conflict_maxspeed_df[
    conflict_maxspeed_df["tags.maxspeed"] != combine_ways_df["max_velocity"]
]

C:\Users\Asriel\AppData\Local\Temp\ipykernel_14496\2497806692.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conflict_maxspeed_df["tags.maxspeed"] = conflict_maxspeed_df["tags.maxspeed"].astype(float)
C:\Users\Asriel\AppData\Local\Temp\ipykernel_14496\2497806692.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conflict_maxspeed_df["max_velocity"] = conflict_maxspeed_df["max_velocity"].astype(float)


,id,tags.maxspeed,max_velocity
801,227746179,40.0,-1.0
1318,431068520,40.0,-1.0


Do chỉ bị mâu thuẫn ở max velocity sẽ sử dụng max_velocity, áp dụng theo các quy tắc của luật VN

In [48]:
def max_velocity_rules(row: pd.Series):
    if int(row["max_velocity"]) == -1:
        # Đường cao tốc
        if row["type"] == "motorway":
            return 120

        oneway = row["tags.oneway"] == "yes"
        if oneway:
            return 50
        else:
            return 60
    return row["max_velocity"]

In [49]:
combine_ways_df["max_velocity"] = combine_ways_df["max_velocity"].astype(float)
combine_ways_df["max_velocity"] = combine_ways_df.apply(max_velocity_rules, axis=1)

In [50]:
combine_ways_df["max_velocity"].unique()

array([ 60.,  40.,  50.,  20.,  80.,  45.,  10.,  70., 120.,  30., 100.])

#### Min speed

In [51]:
combine_ways_df["tags.minspeed"].unique()

array([nan, '60'], dtype=object)

In [52]:
combine_ways_df[combine_ways_df["tags.minspeed"].notna()].shape

(3, 133)

In [53]:
combine_ways_df["tags.minspeed"] = combine_ways_df["tags.minspeed"].fillna(0)

In [54]:
combine_ways_df["tags.access"].unique()

array([nan, 'yes', 'permissive', 'private', 'no'], dtype=object)

#### Motorroad

In [55]:
combine_ways_df["tags.motorroad"].unique()

array([nan, 'yes'], dtype=object)

In [56]:
combine_ways_df[combine_ways_df["tags.motorroad"].notna()].shape

(75, 133)

In [57]:
combine_ways_df["tags.motorroad"] = combine_ways_df["tags.motorroad"].fillna("no")

In [58]:
for tag in way_control_tags:
    node_perc = (combine_ways_df[tag].value_counts(normalize=True) * 100).round(2)
    print(node_perc)
    print("_"*50)

tags.minspeed
0     99.85
60     0.15
Name: proportion, dtype: float64
__________________________________________________
tags.motorroad
no     96.19
yes     3.81
Name: proportion, dtype: float64
__________________________________________________


In [59]:
way_oh_encoder = OneHotEncoder(
    drop="first",     
    sparse_output=False
)
way_oh_tags = ["tags.surface", "tags.oneway", "tags.motorroad", "type"]
way_encoded_array = way_oh_encoder.fit_transform(combine_ways_df[way_oh_tags])
way_oh_encoder.categories_

[array(['asphalt', 'concrete', 'dirt', 'paved', 'unpaved'], dtype=object),
 array(['no', 'yes'], dtype=object),
 array(['no', 'yes'], dtype=object),
 array(['motorway', 'motorway_link', 'primary', 'primary_link',
        'secondary', 'secondary_link', 'tertiary', 'tertiary_link',
        'trunk', 'trunk_link', 'unclassified'], dtype=object)]

In [60]:
way_oh_encoded_df = pd.DataFrame(
    way_encoded_array, 
    columns=way_oh_encoder.get_feature_names_out(),
    index=combine_ways_df.index
)

way_oh_encoded_df.insert(0, "id", combine_ways_df["id"])
way_oh_encoded_df.head()

,id,tags.surface_concrete,tags.surface_dirt,tags.surface_paved,tags.surface_unpaved,tags.oneway_yes,tags.motorroad_yes,type_motorway_link,type_primary,type_primary_link,type_secondary,type_secondary_link,type_tertiary,type_tertiary_link,type_trunk,type_trunk_link,type_unclassified
0,32575820,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,32575862,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,32575864,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,32575869,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,32575935,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Segment

In [61]:
segment_oh_encoder = OneHotEncoder(
    drop="first",     
    sparse_output=False
)
segment_oh_tags = ["segment_type"]
segment_encoded_array = segment_oh_encoder.fit_transform(segments_df[segment_oh_tags])
segment_oh_encoder.categories_

[array(['bank', 'bus_station', 'car', 'cinema', 'clothes', 'company',
        'convenience', 'fuel', 'government', 'house', 'marketplace',
        'motorway', 'motorway_link', 'pitch', 'primary', 'primary_link',
        'residential', 'school', 'secondary', 'secondary_link', 'tertiary',
        'tertiary_link', 'trunk', 'trunk_link', 'unclassified',
        'university'], dtype=object)]

In [62]:
segment_oh_encoded_df = pd.DataFrame(
    segment_encoded_array, 
    columns=segment_oh_encoder.get_feature_names_out(),
    index=segments_df.index
)

segment_oh_encoded_df.insert(0, "_id", segments_df["_id"])
segment_oh_encoded_df.head()

,_id,segment_type_bus_station,segment_type_car,segment_type_cinema,segment_type_clothes,segment_type_company,segment_type_convenience,segment_type_fuel,segment_type_government,segment_type_house,...,segment_type_residential,segment_type_school,segment_type_secondary,segment_type_secondary_link,segment_type_tertiary,segment_type_tertiary_link,segment_type_trunk,segment_type_trunk_link,segment_type_unclassified,segment_type_university
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [63]:
segment_static_df = segments_df.merge(
    segment_oh_encoded_df,
    how="inner",
    on="_id"
)
segment_static_df.head()

,_id,street_id,s_node_id,e_node_id,length,segment_type,segment_type_bus_station,segment_type_car,segment_type_cinema,segment_type_clothes,...,segment_type_residential,segment_type_school,segment_type_secondary,segment_type_secondary_link,segment_type_tertiary,segment_type_tertiary_link,segment_type_trunk,segment_type_trunk_link,segment_type_unclassified,segment_type_university
0,0,32575820,366428456,366416066,0.116,tertiary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1,32575862,366469460,3792257828,0.026,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,32575862,366469460,3792257828,0.026,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,32575862,366403668,5755066033,0.007,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,32575862,366403668,5755066033,0.007,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Viết metadata

In [64]:
OH_dict = dict()
OH_dict["method"] = "one-hot encoding k-1"

In [65]:
OH_dict["node"] = {
    "col": node_attr_tags,
    "features": node_oh_encoder.get_feature_names_out().tolist()
}
OH_dict

{'method': 'one-hot encoding k-1',
 'node': {'col': ['tags.railway',
   'tags.junction',
   'tags.crossing',
   'tags.highway',
   'tags.bus'],
  'features': ['tags.railway_level_crossing',
   'tags.railway_no',
   'tags.railway_station',
   'tags.junction_yes',
   'tags.crossing_traffic_signals',
   'tags.crossing_zebra',
   'tags.highway_crossing',
   'tags.highway_no',
   'tags.highway_traffic_signals',
   'tags.bus_yes']}}

In [66]:
OH_dict["way"] = {
    "col": way_oh_tags,
    "features": way_oh_encoder.get_feature_names_out().tolist()
}
OH_dict

{'method': 'one-hot encoding k-1',
 'node': {'col': ['tags.railway',
   'tags.junction',
   'tags.crossing',
   'tags.highway',
   'tags.bus'],
  'features': ['tags.railway_level_crossing',
   'tags.railway_no',
   'tags.railway_station',
   'tags.junction_yes',
   'tags.crossing_traffic_signals',
   'tags.crossing_zebra',
   'tags.highway_crossing',
   'tags.highway_no',
   'tags.highway_traffic_signals',
   'tags.bus_yes']},
 'way': {'col': ['tags.surface', 'tags.oneway', 'tags.motorroad', 'type'],
  'features': ['tags.surface_concrete',
   'tags.surface_dirt',
   'tags.surface_paved',
   'tags.surface_unpaved',
   'tags.oneway_yes',
   'tags.motorroad_yes',
   'type_motorway_link',
   'type_primary',
   'type_primary_link',
   'type_secondary',
   'type_secondary_link',
   'type_tertiary',
   'type_tertiary_link',
   'type_trunk',
   'type_trunk_link',
   'type_unclassified']}}

In [67]:
OH_dict["segment"] = {
    "col": segment_oh_tags,
    "features": segment_oh_encoder.get_feature_names_out().tolist()
}
OH_dict

{'method': 'one-hot encoding k-1',
 'node': {'col': ['tags.railway',
   'tags.junction',
   'tags.crossing',
   'tags.highway',
   'tags.bus'],
  'features': ['tags.railway_level_crossing',
   'tags.railway_no',
   'tags.railway_station',
   'tags.junction_yes',
   'tags.crossing_traffic_signals',
   'tags.crossing_zebra',
   'tags.highway_crossing',
   'tags.highway_no',
   'tags.highway_traffic_signals',
   'tags.bus_yes']},
 'way': {'col': ['tags.surface', 'tags.oneway', 'tags.motorroad', 'type'],
  'features': ['tags.surface_concrete',
   'tags.surface_dirt',
   'tags.surface_paved',
   'tags.surface_unpaved',
   'tags.oneway_yes',
   'tags.motorroad_yes',
   'type_motorway_link',
   'type_primary',
   'type_primary_link',
   'type_secondary',
   'type_secondary_link',
   'type_tertiary',
   'type_tertiary_link',
   'type_trunk',
   'type_trunk_link',
   'type_unclassified']},
 'segment': {'col': ['segment_type'],
  'features': ['segment_type_bus_station',
   'segment_type_car',
  

In [68]:
fill_dict = dict()
fill_dict["node"] = {
    k: "no" for k in node_attr_tags
}
fill_dict

{'node': {'tags.railway': 'no',
  'tags.junction': 'no',
  'tags.crossing': 'no',
  'tags.highway': 'no',
  'tags.bus': 'no'}}

In [69]:
fill_dict["way"] = {
    "tags.lanes": "(street_type == \"unclassified\") ? 1 : 2",
    "tags.maxspeed": "(street_type == \"motorway\") ? 120 : ((oneway == \"yes\") ? 50 : 60)",
    "tags.minspeed": 0,
    "tags.layer": 0,
    "tags.surface": "(street_type == \"unclassified\") ? \"unpaved\" : \"paved\""
}
fill_dict

{'node': {'tags.railway': 'no',
  'tags.junction': 'no',
  'tags.crossing': 'no',
  'tags.highway': 'no',
  'tags.bus': 'no'},
 'way': {'tags.lanes': '(street_type == "unclassified") ? 1 : 2',
  'tags.maxspeed': '(street_type == "motorway") ? 120 : ((oneway == "yes") ? 50 : 60)',
  'tags.minspeed': 0,
  'tags.layer': 0,
  'tags.surface': '(street_type == "unclassified") ? "unpaved" : "paved"'}}

In [70]:
metadata = {
    "one-hot encoding": OH_dict,
    "fill null": fill_dict
}
metadata

{'one-hot encoding': {'method': 'one-hot encoding k-1',
  'node': {'col': ['tags.railway',
    'tags.junction',
    'tags.crossing',
    'tags.highway',
    'tags.bus'],
   'features': ['tags.railway_level_crossing',
    'tags.railway_no',
    'tags.railway_station',
    'tags.junction_yes',
    'tags.crossing_traffic_signals',
    'tags.crossing_zebra',
    'tags.highway_crossing',
    'tags.highway_no',
    'tags.highway_traffic_signals',
    'tags.bus_yes']},
  'way': {'col': ['tags.surface', 'tags.oneway', 'tags.motorroad', 'type'],
   'features': ['tags.surface_concrete',
    'tags.surface_dirt',
    'tags.surface_paved',
    'tags.surface_unpaved',
    'tags.oneway_yes',
    'tags.motorroad_yes',
    'type_motorway_link',
    'type_primary',
    'type_primary_link',
    'type_secondary',
    'type_secondary_link',
    'type_tertiary',
    'type_tertiary_link',
    'type_trunk',
    'type_trunk_link',
    'type_unclassified']},
  'segment': {'col': ['segment_type'],
   'features':

In [71]:
with open("../metadata/meta.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(metadata, f, sort_keys=False)

# Tổng hợp lại và lưu

In [72]:
node_oh_encoded_df.head()

,id,tags.railway_level_crossing,tags.railway_no,tags.railway_station,tags.junction_yes,tags.crossing_traffic_signals,tags.crossing_zebra,tags.highway_crossing,tags.highway_no,tags.highway_traffic_signals,tags.bus_yes
0,366367392,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,366367451,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,366367839,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,366367847,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,366368010,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [73]:
way_num_df = combine_ways_df[["id", "tags.layer", "tags.lanes", "tags.minspeed", "max_velocity"]]
way_num_df.head()

,id,tags.layer,tags.lanes,tags.minspeed,max_velocity
0,32575820,0,2,0,60.0
1,32575862,0,2,0,60.0
2,32575864,0,2,0,40.0
3,32575869,0,1,0,60.0
4,32575935,0,2,0,50.0


In [74]:
way_features_df = way_oh_encoded_df.merge(
    way_num_df,
    how="inner",
    on="id"
)
print(way_features_df.shape)
way_features_df.head()

(1967, 21)


,id,tags.surface_concrete,tags.surface_dirt,tags.surface_paved,tags.surface_unpaved,tags.oneway_yes,tags.motorroad_yes,type_motorway_link,type_primary,type_primary_link,...,type_secondary_link,type_tertiary,type_tertiary_link,type_trunk,type_trunk_link,type_unclassified,tags.layer,tags.lanes,tags.minspeed,max_velocity
0,32575820,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2,0,60.0
1,32575862,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,2,0,60.0
2,32575864,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,2,0,40.0
3,32575869,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0,1,0,60.0
4,32575935,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,2,0,50.0


In [75]:
node_oh_encoded_df.to_csv("../data/preprocess/static_nodes.csv", index=False)

In [76]:
way_num_df.to_csv("../data/preprocess/static_ways.csv", index=False)

In [78]:
segment_static_df.to_csv("../data/preprocess/static_segments.csv", index=False)

,_id,street_id,s_node_id,e_node_id,length,segment_type,segment_type_bus_station,segment_type_car,segment_type_cinema,segment_type_clothes,...,segment_type_residential,segment_type_school,segment_type_secondary,segment_type_secondary_link,segment_type_tertiary,segment_type_tertiary_link,segment_type_trunk,segment_type_trunk_link,segment_type_unclassified,segment_type_university
0,0,32575820,366428456,366416066,0.116,tertiary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1,32575862,366469460,3792257828,0.026,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,32575862,366469460,3792257828,0.026,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,32575862,366403668,5755066033,0.007,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,32575862,366403668,5755066033,0.007,secondary,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33436,33436,654864528,411918572,1226517262,0.013,primary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33437,33437,654864530,4597281310,411918528,0.126,tertiary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
33438,33438,654864530,4597281310,411918528,0.126,tertiary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
33439,33439,654864530,411918528,411925985,0.089,tertiary,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
